# cuOpt Agent

The cuOpt Agent is a reference optimization assistant that translates natural-language problem descriptions into mathematical models, solves them on the GPU with NVIDIA cuOpt, and returns explained results. It ships with a multi-period supply chain planning scenario (max-supply) as its built-in use case, but the architecture is designed to be extended to other LP/MILP domains by adding new skills, data files, and configs. It is built as a LangChain Deep Agents workflow on top of NAT, with structured skills that make the path from problem text to correct formulation more reliable. The agent uses an orchestrator-subagent pattern: the orchestrator receives user queries and delegates optimization work to a specialized cuOpt subagent via the task() tool, keeping coordination logic separate from problem-solving logic.

Key features:

- GPU-accelerated solver: cuOpt LP/MILP solver for fast optimization on NVIDIA GPUs.
- Structured skills: Parsing, modeling, debugging, and supply-chain planning skills give the agent rules and reference models so it formulates correctly on the first try.
- Mandatory ambiguity handling: When problem text is ambiguous, the agent must clarify or solve all plausible interpretations.
- YAML-driven configuration: Agents, LLMs, and workflows are defined in config files; tune behavior without code changes.
- Web UI: Chat interface for interacting with the agent.
- Tracing: Phoenix and LangSmith (optional) for inspecting agent behavior.
- Evaluation harness: Built-in eval configs for measuring agent quality.

### Software Components

| Component | Purpose |
|-----------|---------|
| [NVIDIA NeMo Agent Toolkit](https://docs.nvidia.com/nemo/agent-toolkit/latest/) | Agent framework and serving |
| [NVIDIA cuOpt](https://developer.nvidia.com/cuopt) | GPU-accelerated LP/MILP solver |
| [LangChain Deep Agents](https://www.langchain.com/) | Multi-step agent workflow |
| [minimaxai/minimax-m2.5](https://build.nvidia.com/) (via NIM) | LLM for agent reasoning |
| [Phoenix](https://github.com/Arize-ai/phoenix) | OpenTelemetry tracing |
| [LangSmith](https://smith.langchain.com/) (optional) | Agent tracing and observability |
| [NAT UI](external/nat-ui/) | Chat web interface |


### Prerequisites

- NVIDIA GPU with CUDA support
- Docker and Docker Compose (with [NVIDIA Container Toolkit](https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html))
- Access to `nvcr.io/nvidia/cuopt/cuopt:26.2.0-cuda12.9-py3.13` base image ([NGC](https://catalog.ngc.nvidia.com/))
- NVIDIA API key from [build.nvidia.com](https://build.nvidia.com/)

Since you are running this in a Brev launchable environment, CUDA and Docker prereqs are taken care of. Before running through this notebook, ensure you have access to the cuOpt image and that you have an API key. 

**Optional:**

- [LangSmith](https://smith.langchain.com/) API key for agent tracing (uncomment the LangSmith section in the config YAML to enable)


**For local development (without Docker) and improved performance**
Check the guide [here](https://github.com/NVIDIA/cuopt-examples/tree/main/cuopt-agent#readme)

## 1. Clone the Repository
First, we will clone the cuOpt examples repostory, which has sample notebooks for a wide variety of use cases. The subdirectory `cuopt-agent` includes all the code needed for this demo. 
Let's clone the repo and move into the relevant subdirectory.

In [ ]:
!git clone https://github.com/NVIDIA/cuopt-examples.git

In [ ]:
%cd cuopt-examples/cuopt-agent

## 2. Quick Start
Let's initialize a couple submodules and environment variables.

In [ ]:
# paste here your API key from build.nvidia.com
import os
os.environ['NVIDIA_API_KEY'] = 'nvapi-***'

In [ ]:
# 1. Initialize submodules (required for the Chat UI)
! git submodule update --init --recursive

# 2. Set up environment variables
!cp .env.example .env

# save the API Key to the .env file
with open(".env", "a") as f:
    f.write(f"\nNVIDIA_API_KEY={os.environ['NVIDIA_API_KEY']}\n")



In [ ]:
# log in to NVIDIA container registry so we can access the cuOpt container
! docker login nvcr.io --username '$oauthtoken' --password {os.environ['NVIDIA_API_KEY']}

## 3. Build and start all services

We are ready to build and run the demo. Note that each one of the following commands may take a couple minutes to run. 

In [ ]:
! docker compose -f deploy/compose/docker-compose.yml build

In [ ]:
! docker compose -f deploy/compose/docker-compose.yml up -d

## 4. open the Chat UI and Phoenix Tracing. 

Go back to the Brev Launchable page, and in the `Using Secure Links` sections add port 3000 for Chat UI and port 6006 for Phoenix tracing. Then click the arrow buttons to launch in browser.

There are sample prompts you can copy and paste in the UI in the [`cuopt-examples' repo](https://github.com/NVIDIA/cuopt-examples/tree/main/cuopt-agent/cuopt_agent/data/max_supply_what_ifs/sample_prompts)

